# RAG Module · Day 22 — **GraphRAG & Agentic Retrieval**

**Phase 1 · Module 3: Prompt Engineering & RAG Architecture** · ~2 hrs

Yesterday's hybrid search retrieves the *k* chunks most similar to a query. But some questions have no single chunk that answers them — *"Which product does Sarah's account use, and what's its overdraft limit?"* needs **two hops**: Sarah → her account → that product → its limit. Vector search returns the chunk nearest the words and stops. Two answers to multi-hop:

1. **GraphRAG** — extract entities & relations into a **knowledge graph**, then *traverse* it.
2. **Agentic retrieval** — an agent that retrieves, *reasons about the gap*, and retrieves **again**.

Today:
1. See naive vector RAG **fail** a multi-hop question.
2. Extract entities → build a graph store → answer by traversal.
3. Build an agentic retriever that loops retrieve→reason→retrieve.
4. Compare all three on multi-hop questions.

> Pure Python, stdlib only. No keys. Entity extraction is rule-based here; in production it's an LLM extraction prompt (or Microsoft GraphRAG) — the *retrieval* logic is identical.

## 1. Naive vector RAG hits a wall

A small set of banking facts as chunks. The query needs two facts joined. Vector search returns the single closest chunk — which holds only *half* the answer.

In [1]:
import re
from collections import defaultdict, deque

FACTS = [
    "Sarah Khan holds account AC-1001.",
    "Account AC-1001 is a Premier Current Account.",
    "The Premier Current Account has an overdraft limit of 3000 pounds.",
    "The Premier Current Account charges no monthly fee.",
    "James Patel holds account AC-1002.",
    "Account AC-1002 is a Student Current Account.",
    "The Student Current Account has an overdraft limit of 1500 pounds.",
    "Account AC-1001 was opened at the Leeds branch.",
    "The Leeds branch is managed by Priya Nair.",
]

def tok(s): return set(re.findall(r"[a-z0-9]+", s.lower()))

def vector_rag(query, k=1):
    q = tok(query)
    ranked = sorted(FACTS, key=lambda f: len(q & tok(f)), reverse=True)
    return ranked[:k]

q = "What overdraft limit applies to Sarah Khan's account?"
print("Q:", q)
for hit in vector_rag(q, k=2):
    print("  retrieved:", hit)

Q: What overdraft limit applies to Sarah Khan's account?
  retrieved: Sarah Khan holds account AC-1001.
  retrieved: The Premier Current Account has an overdraft limit of 3000 pounds.


☝ The top chunks match on *"Sarah Khan"* / *"account"* / *"overdraft"* individually, but **no single chunk** links Sarah → AC-1001 → Premier → 3000. Vector RAG can't *join* facts; it retrieves the nearest one and hands the LLM half the answer. Stuffing more k just adds noise, not the connection.

## 2. Extract entities → knowledge graph

GraphRAG parses each fact into **(subject, relation, object)** triples and stores them as a graph: nodes are entities, edges are relations. Then a multi-hop question becomes a **path** through the graph. Extraction here is regex rules; production swaps in an LLM extraction prompt — the graph and traversal stay the same.

In [2]:
# rule-based triple extraction (LLM prompt in production)
RULES = [
    (r"(.+?) holds account (\S+)",                    "holds"),
    (r"account (\S+) is an? (.+)",                    "is_a"),
    (r"(?:the )?(.+?) has an overdraft limit of (.+)", "overdraft_limit"),
    (r"(?:the )?(.+?) charges (.+)",                   "fee"),
    (r"account (\S+) was opened at the (.+)",         "opened_at"),
    (r"(?:the )?(.+?) is managed by (.+)",             "managed_by"),
]
def norm(s): return s.strip().strip(".").lower()

edges = defaultdict(list)          # subj -> list of (relation, obj)
triples = []
for f in FACTS:
    for pat, rel in RULES:
        m = re.match(pat, f.lower())
        if m:
            s, o = norm(m.group(1)), norm(m.group(2))
            edges[s].append((rel, o)); triples.append((s, rel, o))
            break

print(f"extracted {len(triples)} triples. sample:")
for t in triples[:5]: print("  ", t)
print("\nedges from 'sarah khan':", edges["sarah khan"])

extracted 9 triples. sample:
   ('sarah khan', 'holds', 'ac-1001')
   ('ac-1001', 'is_a', 'premier current account')
   ('premier current account', 'overdraft_limit', '3000 pounds')
   ('premier current account', 'fee', 'no monthly fee')
   ('james patel', 'holds', 'ac-1002')

edges from 'sarah khan': [('holds', 'ac-1001')]


☝ Free text is now a graph: `sarah khan --holds--> ac-1001 --is_a--> premier current account --overdraft_limit--> 3000 pounds`. The join that vector RAG couldn't make is an explicit chain of edges — and traversing it is a graph walk, not a similarity score.

## 3. Answer by graph traversal

Find the entity the question anchors on, then **walk** edges to the target relation. This BFS follows the chain across as many hops as needed — the capability naive RAG lacks.

In [3]:
def find_entity(query):
    q = tok(query)
    best, score = None, 0
    for node in edges:
        s = len(q & tok(node))
        if s > score: best, score = node, s
    return best

def graph_answer(query, target_rel):
    start = find_entity(query)
    seen, dq = set(), deque([(start, [start])])
    while dq:
        node, path = dq.popleft()
        if node in seen: continue
        seen.add(node)
        for rel, obj in edges.get(node, []):
            if rel == target_rel:
                return obj, path + [obj]
            dq.append((obj, path + [obj]))
    return None, [start]

ans, path = graph_answer("Sarah Khan's account overdraft", "overdraft_limit")
print("anchor entity :", find_entity("Sarah Khan's account overdraft"))
print("answer        :", ans)
print("reasoning path:", " -> ".join(path))

# 3-hop: Sarah -> account -> branch -> branch manager
ans2, path2 = graph_answer("who manages the branch where Sarah opened her account", "managed_by")
print("\n3-hop answer :", ans2, "| path:", " -> ".join(path2))

anchor entity : sarah khan
answer        : 3000 pounds
reasoning path: sarah khan -> ac-1001 -> premier current account -> 3000 pounds

3-hop answer : priya nair | path: sarah khan -> ac-1001 -> leeds branch -> priya nair


☝ Traversal from `sarah khan` walks `holds → is_a → overdraft_limit` and returns **3000 pounds** with the full path as an audit trail — and the same walk answers a *3-hop* question (Sarah → account → Leeds branch → Priya Nair). GraphRAG shines exactly where relationships chain: KYC, transaction networks, org hierarchies.

## 4. Agentic retrieval — retrieve, reason, retrieve again

You don't always have a pre-built graph. **Agentic retrieval** gets multi-hop from the *agent loop*: retrieve for the current sub-question, read what's still missing, form the next sub-question, retrieve again — until it can answer. This is AWS's **Bedrock Agentic Retriever** pattern for complex queries.

In [4]:
def retrieve_unseen(subq, known):
    q = tok(subq)
    pool = [f for f in FACTS if f not in known]
    return max(pool, key=lambda f: len(q & tok(f))) if pool else None

def agentic_retrieve(query, max_hops=3):
    known, trace = [], []
    # seed sub-question: the entity in the original query
    subq = find_entity(query) or query
    for hop in range(max_hops):
        hit = retrieve_unseen(subq, known)
        if hit is None: break
        known.append(hit); trace.append((subq, hit))
        # reason: does 'known' contain the overdraft figure yet?
        joined = " ".join(known)
        m = re.search(r"overdraft limit of (.+?)\.", joined)
        if m:
            return m.group(1), trace
        # gap: we have an account or product but not its limit -> retrieve the next link
        acc = re.search(r"(AC-\d+)", hit)
        prod = re.search(r"is an? (.+?)\.", hit)
        subq = (prod.group(1) if prod else acc.group(1) if acc else hit)
    return None, trace

ans, trace = agentic_retrieve("What overdraft limit applies to Sarah Khan's account?")
print("agentic answer:", ans, "\n")
for i, (sq, hit) in enumerate(trace, 1):
    print(f"  hop {i}: asked {sq!r}\n         got  {hit}")

agentic answer: 3000 pounds 

  hop 1: asked 'sarah khan'
         got  Sarah Khan holds account AC-1001.
  hop 2: asked 'AC-1001'
         got  Account AC-1001 is a Premier Current Account.
  hop 3: asked 'Premier Current Account'
         got  The Premier Current Account has an overdraft limit of 3000 pounds.


☝ The agent needed three retrievals: *Sarah* → `AC-1001`, then *AC-1001* → *Premier Current Account*, then *Premier* → *3000 pounds*. Each hop's result **reshapes the next query** — no pre-built graph required, at the cost of multiple LLM+retrieval round-trips. That's the trade: GraphRAG pays once at ingestion, agentic retrieval pays per query.

## 5. Compare — standard vs graph vs agentic

In [5]:
QS = [
    ("What overdraft limit applies to Sarah Khan's account?", "overdraft_limit", "3000 pounds"),
    ("What overdraft limit applies to James Patel's account?", "overdraft_limit", "1500 pounds"),
]
print(f"{'question':52} {'vector':8} {'graph':8} {'agentic':8}")
for q, rel, gold in QS:
    v = vector_rag(q, 1)[0]
    v_ok = gold.split()[0] in v                       # does single chunk contain the number?
    g_ok = graph_answer(q, rel)[0] == gold
    a_ok = (agentic_retrieve(q)[0] or "").strip() == gold
    tick = lambda b: "  ✓  " if b else "  ✗  "
    print(f"{q[:52]:52}{tick(v_ok)}   {tick(g_ok)}   {tick(a_ok)}")
print("\n✓ = returned the correct linked answer")

question                                             vector   graph    agentic 
What overdraft limit applies to Sarah Khan's account  ✗       ✓       ✓  
What overdraft limit applies to James Patel's accoun  ✗       ✓       ✓  

✓ = returned the correct linked answer


☝ Naive vector RAG fails both — the joining fact lives in no single chunk. **Graph** and **agentic** both succeed: one by traversing pre-built edges, the other by looping retrieval. Pick by workload — a stable, relationship-heavy domain (accounts, KYC, org charts) favours GraphRAG's ingest-time graph; ad-hoc multi-hop questions over changing docs favour the agentic loop.

## 6. Recap — when nearest-chunk isn't enough

| Approach | How multi-hop works | Cost | Best for |
|---|---|---|---|
| **Vector RAG** | can't — returns nearest single chunk | cheap | single-fact lookups |
| **GraphRAG** | traverse entity→relation edges | build graph once at ingest | stable, relationship-heavy domains |
| **Agentic retrieval** | loop retrieve→reason→retrieve | many round-trips per query | ad-hoc multi-hop over changing docs |

Multi-hop is where RAG stops being similarity search and becomes **reasoning over structure**. On Bedrock: GraphRAG via a graph store, agentic via the **Agentic Retriever**. 